# CovidScope — Interactive Analysis

This notebook walks through the full COVID-19 analysis pipeline:
1. Load & inspect the raw data
2. Clean, merge and engineer features
3. Explore country summaries and peaks
4. Generate all visualisations

> Run the cells in order. Charts are saved to `../visualizations/`.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure we can import the analysis module
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root / "src"))

from analysis import (
    load_data,
    clean_and_merge,
    country_summary,
    detect_peaks,
    generate_visualizations,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", palette="colorblind")
print("Setup complete.")

## 2. Load raw data

In [ ]:
cases, deaths, vaccinations = load_data()

print("Cases shape:", cases.shape)
print("Deaths shape:", deaths.shape)
print("Vaccinations shape:", vaccinations.shape)
print()
cases.head()

In [ ]:
deaths.head()

In [ ]:
vaccinations.head()

## 3. Clean, merge & engineer features

In [ ]:
df = clean_and_merge(cases, deaths, vaccinations)
print("Merged shape:", df.shape)
print("\nColumns:", list(df.columns))
df.head(10)

In [ ]:
# Quick data-quality check
print("Missing values:\n", df.isna().sum())
print("\nDate range:", df["date"].min().date(), "→", df["date"].max().date())
print("Countries:", sorted(df["country"].unique()))

## 4. Country summary & rankings

In [ ]:
summary = country_summary(df)
summary

## 5. Peak detection

In [ ]:
print("Highest case peaks:")
display(detect_peaks(df, value_col="cases", top_n=4))

print("\nHighest death peaks:")
display(detect_peaks(df, value_col="deaths", top_n=4))

## 6. Generate all visualisations

Running the cell below produces (or refreshes) every chart in `../visualizations/`.

In [ ]:
df = generate_visualizations()

## 7. Quick custom exploration

Feel free to add your own cells below. Example: vaccination coverage vs CFR.

In [ ]:
latest = df.sort_values("date").groupby("country", as_index=False).last()

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=latest,
    x="people_vaccinated_pct",
    y="cfr",
    hue="country",
    s=140,
    ax=ax,
)
ax.set_title("Latest Vaccination Coverage vs Case-Fatality Rate")
ax.set_xlabel("People vaccinated (%)")
ax.set_ylabel("CFR (%)")
plt.tight_layout()
plt.show()

---
*End of notebook. All charts are ready in the `visualizations/` folder.*